# **Stratified Sampling untuk Anotasi (incremental, anti-duplikat)**

Mengambil sampel **stratified per aspek** dari `sentences_with_aspect.csv` untuk dianotasi sentimennya.

Desain **incremental**: bisa dijalankan berkali-kali untuk menambah batch tanpa memilih ulang kalimat yang sudah pernah terpilih. Kuncinya:
- setiap kalimat punya `id_kalimat` unik;
- daftar id yang sudah terpilih disimpan di `data/annotation/sudah_terpilih.csv`;
- batch baru selalu **mengecualikan** id yang sudah ada -> tidak ada duplikat;
- `random_state` tetap -> reproducible.

**Lainnya** (outlier/topik tak jelas) dikeluarkan dari sampling.

In [17]:
from pathlib import Path
import math
import pandas as pd

SEED = 42

TARGET_TOTAL = 15000
N_OVERLAP = 150
MIN_PER_ASPEK = 80

ANNDIR = Path("data/annotation")
ANNDIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    "data/processed/sentences_with_aspect.csv",
    dtype={"id_artikel": str, "id_kalimat": str}
)

# Kandidat anotasi: semua kalimat kecuali aspek Lainnya
kandidat = df[df["nama_aspek"] != "Lainnya"].copy()

assert kandidat["id_kalimat"].is_unique, "id_kalimat harus unik!"
assert kandidat["id_kalimat"].notna().all(), "Ada id_kalimat kosong!"

print(f"Total data       : {len(df):,}")
print(f"Kandidat anotasi : {len(kandidat):,}")
print(f"Jumlah aspek     : {kandidat['nama_aspek'].nunique()}")

Total data       : 63,632
Kandidat anotasi : 39,561
Jumlah aspek     : 15


In [18]:
if len(kandidat) < TARGET_TOTAL:
    raise ValueError(
        f"Kandidat hanya {len(kandidat):,}, "
        f"sedangkan target {TARGET_TOTAL:,}."
    )

if N_OVERLAP >= TARGET_TOTAL:
    raise ValueError("N_OVERLAP harus lebih kecil dari TARGET_TOTAL.")

jumlah_aspek = kandidat["nama_aspek"].nunique()

if TARGET_TOTAL < MIN_PER_ASPEK * jumlah_aspek:
    raise ValueError(
        "TARGET_TOTAL terlalu kecil untuk memenuhi MIN_PER_ASPEK."
    )

print(f"Target kalimat unik : {TARGET_TOTAL:,}")
print(f"Target overlap     : {N_OVERLAP:,}")
print(f"Target non-overlap : {TARGET_TOTAL - N_OVERLAP:,}")

Target kalimat unik : 15,000
Target overlap     : 150
Target non-overlap : 14,850


In [19]:
ukuran = kandidat["nama_aspek"].value_counts().sort_index()

# Alokasi awal: minimum per aspek, dibatasi ketersediaan
alok = {
    aspek: min(MIN_PER_ASPEK, int(jumlah))
    for aspek, jumlah in ukuran.items()
}

sisa_target = TARGET_TOTAL - sum(alok.values())

# Distribusikan sisa target secara proporsional
while sisa_target > 0:

    aktif = [
        aspek for aspek in ukuran.index
        if alok[aspek] < ukuran[aspek]
    ]

    if not aktif:
        raise ValueError("Kandidat tidak cukup untuk memenuhi target.")

    bobot = ukuran.loc[aktif]
    proporsi = sisa_target * bobot / bobot.sum()

    tambahan_dasar = {
        aspek: min(
            int(math.floor(proporsi[aspek])),
            int(ukuran[aspek] - alok[aspek])
        )
        for aspek in aktif
    }

    jumlah_tambahan = sum(tambahan_dasar.values())

    for aspek, tambahan in tambahan_dasar.items():
        alok[aspek] += tambahan

    sisa_target -= jumlah_tambahan

    if sisa_target == 0:
        break

    # Distribusikan sisa pembulatan berdasarkan fractional remainder
    ranking = sorted(
        aktif,
        key=lambda aspek: (
            -(proporsi[aspek] - math.floor(proporsi[aspek])),
            aspek
        )
    )

    for aspek in ranking:
        if sisa_target == 0:
            break

        if alok[aspek] < ukuran[aspek]:
            alok[aspek] += 1
            sisa_target -= 1

print("Alokasi per aspek:")

for aspek in ukuran.index:
    print(
        f"{aspek:35} "
        f"{alok[aspek]:>6,} / tersedia {ukuran[aspek]:,}"
    )

print("Total alokasi:", sum(alok.values()))

assert sum(alok.values()) == TARGET_TOTAL

Alokasi per aspek:
Aksi Investor Asing                  1,476 / tersedia 4,001
Bursa Global                           460 / tersedia 1,089
Dividen & Aksi Korporasi               973 / tersedia 2,561
Kebijakan Moneter & Nilai Tukar      1,589 / tersedia 4,326
Kinerja Keuangan & Laba                468 / tersedia 1,112
Kredit & Kualitas Aset                 908 / tersedia 2,373
Layanan & Produk                       719 / tersedia 1,832
Makroekonomi & Faktor Global           586 / tersedia 1,451
Manajemen & Kepemimpinan               443 / tersedia 1,039
Pergerakan IHSG & Indeks             2,559 / tersedia 7,107
Pergerakan Saham Bank                1,299 / tersedia 3,494
Saham Spesifik Non-Bank                460 / tersedia 1,089
Sektor & Transaksi Pasar             1,341 / tersedia 3,616
Strategi & Ekspansi Bisnis           1,537 / tersedia 4,178
Valuasi & Kapitalisasi                 182 / tersedia 293
Total alokasi: 15000


In [20]:
bagian_sampel = []

for aspek in ukuran.index:

    n = alok[aspek]

    pool = kandidat[
        kandidat["nama_aspek"] == aspek
    ]

    sampel_aspek = pool.sample(
        n=n,
        random_state=SEED
    )

    bagian_sampel.append(sampel_aspek)

sampel = (
    pd.concat(bagian_sampel, ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

assert len(sampel) == TARGET_TOTAL
assert sampel["id_kalimat"].is_unique

print(f"Jumlah sampel: {len(sampel):,}")
print("Verifikasi ID unik: OK")

Jumlah sampel: 15,000
Verifikasi ID unik: OK


In [21]:
overlap = sampel.sample(
    n=N_OVERLAP,
    random_state=SEED
).copy()

id_overlap = set(overlap["id_kalimat"])

non_overlap = sampel[
    ~sampel["id_kalimat"].isin(id_overlap)
].copy()

non_overlap = non_overlap.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

half = len(non_overlap) // 2

porsi_A = non_overlap.iloc[:half].copy()
porsi_B = non_overlap.iloc[half:].copy()

print("Overlap      :", len(overlap))
print("Khusus A     :", len(porsi_A))
print("Khusus B     :", len(porsi_B))

Overlap      : 150
Khusus A     : 7425
Khusus B     : 7425


In [ ]:
def sheet(frame, anotator):

    out = frame[
        [
            "id_kalimat",
            "id_artikel",
            "bank",
            "timestamp",
            "kalimat",
            "nama_aspek",
            "kategori_fiqa"
        ]
    ].copy()
    out["anotator"] = anotator
    out["sentimen"] = ""
    return out


anotator1 = pd.concat(
    [
        sheet(porsi_A, "A"),
        sheet(overlap, "A")
    ],
    ignore_index=True
)

anotator2 = pd.concat(
    [
        sheet(porsi_B, "B"),
        sheet(overlap, "B")
    ],
    ignore_index=True
)

print(f"Anotator A: {len(anotator1):,}")
print(f"Anotator B: {len(anotator2):,}")

Anotator A: 7,575
Anotator B: 7,575


### **Validasi Duplikat**

In [23]:
ids_A = set(anotator1["id_kalimat"])
ids_B = set(anotator2["id_kalimat"])

irisan_AB = ids_A & ids_B
union_AB = ids_A | ids_B

assert irisan_AB == id_overlap, (
    "Ada data yang sama antara A dan B di luar overlap!"
)

assert len(union_AB) == TARGET_TOTAL
assert len(irisan_AB) == N_OVERLAP

assert anotator1["id_kalimat"].is_unique
assert anotator2["id_kalimat"].is_unique

print("Validasi pembagian data: OK")
print("Total ID unik       :", len(union_AB))
print("ID overlap          :", len(irisan_AB))
print("ID khusus A         :", len(ids_A - ids_B))
print("ID khusus B         :", len(ids_B - ids_A))

Validasi pembagian data: OK
Total ID unik       : 15000
ID overlap          : 150
ID khusus A         : 7425
ID khusus B         : 7425


In [24]:
anotator1.to_csv(
    ANNDIR / "to_annotate_A.csv",
    index=False,
    encoding="utf-8-sig"
)

anotator2.to_csv(
    ANNDIR / "to_annotate_B.csv",
    index=False,
    encoding="utf-8-sig"
)

overlap[["id_kalimat"]].to_csv(
    ANNDIR / "overlap.csv",
    index=False,
    encoding="utf-8-sig"
)

# Simpan master sampel untuk audit/reproducibility
sampel.to_csv(
    ANNDIR / "sampled_15000_master.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File anotasi berhasil disimpan:")
print(ANNDIR / "to_annotate_A.csv")
print(ANNDIR / "to_annotate_B.csv")
print(ANNDIR / "overlap.csv")
print(ANNDIR / "sampled_15000_master.csv")

File anotasi berhasil disimpan:
data\annotation\to_annotate_A.csv
data\annotation\to_annotate_B.csv
data\annotation\overlap.csv
data\annotation\sampled_15000_master.csv
